# Pipeline integrado do hackathon

Este notebook liga as partes do time:

| Parte (branch) | Pasta | Papel aqui |
|---|---|---|
| `chore/add_dataset_files` | `imagens/` | imagens originais |
| `feat/add_imagens_preprocessadas` | `imagens_preprocessadas/` | binarização do time (entrada `binarizada`) |
| `feat/OCR` | `test_kraken/` | motor Kraken (segmentação neural + PP-OCRv6 medium) |
| `llm_implementation` | `LLM_Transcription.ipynb` | motor LLM (OpenRouter) e métricas de avaliação |
| — | `ground_truth/` | gabarito por página, extraído do DOCX |

Cada combinação (`kraken_binarizada`, `llm_original`, `llm_corrige_kraken_binarizada`...) grava `resultados/<nome>.json`, no mesmo formato do notebook do LLM, e todas são avaliadas pelas mesmas métricas. Detalhes no `README.md`.

**Custos:** Kraken é gratuito. As combinações de LLM ficam bloqueadas enquanto `PERMITIR_GASTO = False`; com `SIMULAR = True` as requisições são montadas, mas não enviadas.

Use um ambiente com GPU (**Ambiente de execução → Alterar o tipo de ambiente de execução → GPU**). Em CPU, o Kraken leva cerca de 1 minuto por página e combinação.

In [ ]:
import os
from pathlib import Path

if Path('/content').exists() and not Path('/content/hackathon_ocr_historico').exists():
    !git clone -q https://github.com/fabiolobato/hackathon_ocr_historico.git /content/hackathon_ocr_historico
if Path('/content/hackathon_ocr_historico').exists():
    os.chdir('/content/hackathon_ocr_historico')
print(Path.cwd())

## 1. Dependências

Instala as versões fixadas do `test_kraken` e o `jiwer` das métricas. Se aparecer o aviso de versões antigas carregadas, use **Ambiente de execução → Reiniciar sessão** e execute de novo desde o início.

In [ ]:
import sys
sys.path.insert(0, 'test_kraken')
from kraken_pipeline.setup import configure_notebook, ensure_dependencies

configure_notebook()
ensure_dependencies(Path('test_kraken/requirements.txt'))
%pip -q install jiwer==4.0.0

## 2. Páginas, entradas e motor Kraken

As páginas são ligadas pelo número no nome: imagem original, binarização do time e gabarito. `PAGINAS = None` usa as 12; para um teste rápido, use por exemplo `[11]`.

In [ ]:
import logging
from ocr_historico.dados import listar_paginas
from ocr_historico.combinacoes import interpretar, PADRAO_KRAKEN, PADRAO_LLM
from ocr_historico.executar import executar, avaliar_pasta, imprimir_tabela, PASTA_RESULTADOS
from ocr_historico.motor_kraken import MotorKraken, baixar_modelo
from kraken_pipeline.setup import default_device

logging.getLogger('kraken').setLevel(logging.WARNING)
PAGINAS = None
paginas = listar_paginas(PAGINAS)
print([(p.numero, p.binarizada is not None, p.gabarito is not None) for p in paginas])

motor_kraken = MotorKraken(baixar_modelo('medium'), default_device())

In [ ]:
from IPython.display import display
from ocr_historico.entradas import carregar_entrada

pagina = paginas[min(6, len(paginas) - 1)]
for entrada in ('original', 'binarizada', 'nlbin'):
    print(entrada)
    display(carregar_entrada(pagina, entrada).resize((330, 450)))

## 3. Kraken com cada entrada

Combinações padrão: imagem original, nlbin do Kraken, binarização do time e as versões ampliadas 2×. Os resultados já transcritos são reaproveitados; use `refazer=True` para gerar de novo.

In [ ]:
COMBINACOES_KRAKEN = list(PADRAO_KRAKEN)

for nome in COMBINACOES_KRAKEN:
    print(executar(interpretar(nome), paginas, PASTA_RESULTADOS, motor_kraken=motor_kraken))

## 4. LLM: imagem → LLM e Kraken → LLM

- `llm_original`: o que o notebook `LLM_Transcription.ipynb` já fazia.
- `llm_binarizada`: binarização do time → LLM.
- `llm_corrige_<combinação kraken>`: o LLM recebe a imagem original **e** o texto do Kraken e corrige.

Chave: segredo `API_KEY` do Colab (como no notebook do LLM), variável `OPENROUTER_API_KEY` ou arquivo `.env`. Custo de referência: ~US$ 0,06 por página e combinação.

Se vocês já têm o JSON gerado pelo `LLM_Transcription.ipynb`, importem-no em vez de pagar de novo:

```python
from ocr_historico.resultados import importar_resultado_llm
importar_resultado_llm(Path('resultados_transcricao (2).json'), PASTA_RESULTADOS, 'llm_original')
```

In [ ]:
from ocr_historico.motor_llm import MotorLLM

PERMITIR_GASTO = False  # só mude depois de aprovar o custo
SIMULAR = not PERMITIR_GASTO
COMBINACOES_LLM = list(PADRAO_LLM)

motor_llm = MotorLLM(permitir_gasto=PERMITIR_GASTO, simular=SIMULAR)
pasta_llm = PASTA_RESULTADOS / 'simulacao' if SIMULAR else PASTA_RESULTADOS
for nome in COMBINACOES_LLM:
    print(executar(interpretar(nome), paginas, pasta_llm, motor_llm=motor_llm))
print(f'{len(motor_llm.requisicoes_simuladas)} requisições simuladas (nenhuma enviada)' if SIMULAR else 'enviado')

## 5. Comparação

Mesmas páginas e métricas para todas as combinações em `resultados/`. Valores de corpus somam erros de todas as páginas. `wer_estrito` preserva caixa e pontuação (métrica do `test_kraken`). As demais colunas vêm do notebook do LLM. O gabarito ainda é provisório: veja `ground_truth/comparison_review.md`.

In [ ]:
resumo = avaliar_pasta(PASTA_RESULTADOS, paginas)
imprimir_tabela(resumo)
print('Tabelas completas: resultados/comparacao.csv e resultados/metricas_paginas.csv')

In [ ]:
from ocr_historico.resultados import carregar, textos_por_arquivo

PAGINA, COMBINACOES_VER = pagina, [r['combinacao'] for r in resumo[:3]]
print('GABARITO\n', PAGINA.texto_gabarito(), '\n')
for nome in COMBINACOES_VER:
    print(nome.upper(), '\n', textos_por_arquivo(carregar(PASTA_RESULTADOS / f'{nome}.json')).get(PAGINA.arquivo), '\n')